With robot data, and an additional synthetic customer data, merge the datasets after feature engineering and cleaning.

In [1]:
import pandas as pd

robot_df = pd.read_csv("./data/rmbr4-2_preaugmented.csv")
workshops_df = pd.read_csv("./data/synthetic_robot_workshops.csv")

workshop_lookup = workshops_df[["robot_id", "shop_id", "shop_name"]].drop_duplicates()
if workshop_lookup["robot_id"].duplicated().any():
    raise ValueError("A robot_id is assigned to multiple workshops.")

postaugmented_df = robot_df.merge(
    workshop_lookup,
    on="robot_id",
    how="left",
    validate="many_to_one",
)

if postaugmented_df["shop_id"].isna().any():
    raise ValueError("Some robot_id values do not have a workshop mapping.")

postaugmented_df.to_csv("./data/rmbr4-2_postaugmented.csv", index=False)
print(f"Exported {len(postaugmented_df)} rows to data/rmbr4-2_postaugmented.csv")

Exported 39672 rows to data/rmbr4-2_postaugmented.csv


In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

postaugmented_df = pd.read_csv("./data/rmbr4-2_postaugmented.csv")

axis_columns = [
    column
    for column in postaugmented_df.columns
    if column.startswith("Axis #")
    and pd.to_numeric(postaugmented_df[column], errors="coerce").notna().any()
]
if not axis_columns:
    raise ValueError("The dataset has no numeric Axis columns to plot.")

robot_ids = sorted(postaugmented_df["robot_id"].dropna().unique())
robot_selector = widgets.Dropdown(
    options=[("All robots", None)] + [(str(robot_id), robot_id) for robot_id in robot_ids],
    value=None,
    description="Robot ID:",
)
axis_selector = widgets.Dropdown(
    options=axis_columns,
    value="Axis #1" if "Axis #1" in axis_columns else axis_columns[0],
    description="Column:",
)
graph_selector = widgets.Dropdown(
    options=[("Values", "values"), ("Distribution", "distribution")],
    value="values",
    description="Graph:",
)
plot_button = widgets.Button(description="Show graph", button_style="primary")
plot_output = widgets.Output()


def plot_selected_data(_=None):
    with plot_output:
        plot_output.clear_output(wait=True)
        selected_robot_id = robot_selector.value
        selected_column = axis_selector.value
        selected_data = postaugmented_df
        if selected_robot_id is not None:
            selected_data = selected_data[
                selected_data["robot_id"] == selected_robot_id
            ]

        axis_values = (
            pd.to_numeric(selected_data[selected_column], errors="coerce")
            .dropna()
            .to_numpy()
        )
        if axis_values.size == 0:
            print(f"No numeric values found in {selected_column} for this selection.")
            return

        robot_label = (
            "All robots" if selected_robot_id is None else f"Robot {selected_robot_id}"
        )
        figure, axis = plt.subplots(figsize=(10, 5))
        if graph_selector.value == "distribution":
            mean_value = np.mean(axis_values)
            standard_deviation = np.std(axis_values)
            axis.hist(axis_values, bins="auto", alpha=0.7, edgecolor="white")
            axis.axvspan(
                mean_value - standard_deviation,
                mean_value + standard_deviation,
                alpha=0.18,
                color="orange",
                label=f"Mean ± 1 SD ({standard_deviation:.3g})",
            )
            axis.axvline(
                mean_value,
                color="red",
                linestyle="-",
                linewidth=2,
                label=f"Mean ({mean_value:.3g})",
            )
            axis.set_title(f"{selected_column} Distribution - {robot_label}")
            axis.set_xlabel(selected_column)
            axis.set_ylabel("Frequency")
        else:
            quartile_groups = np.array_split(axis_values, 4)
            start_index = 0
            for quartile_number, values in enumerate(quartile_groups, start=1):
                end_index = start_index + len(values)
                axis.plot(
                    range(start_index, end_index),
                    values,
                    label=f"Q{quartile_number}",
                    linewidth=1,
                )
                if quartile_number < 4:
                    axis.axvline(end_index, color="gray", linestyle="--", linewidth=0.8)
                start_index = end_index

            axis.set_title(f"{selected_column} Values by Quartile - {robot_label}")
            axis.set_xlabel("Observation rank")
            axis.set_ylabel(selected_column)

        axis.legend()
        axis.grid(True, alpha=0.25)
        figure.tight_layout()
        plt.show()


plot_button.on_click(plot_selected_data)
display(
    widgets.HBox([robot_selector, axis_selector, graph_selector]),
    plot_button,
    plot_output,
)
plot_selected_data()

Button(button_style='primary', description='Show graph', style=ButtonStyle())

Output()